# Week 2: Post-Class Exercise SOLUTIONS - Titanic Survival

**Complete solutions with all code filled in and comprehensive sample answers**

---

## Part 1: Setup - SOLUTION

In [ ]:
# SOLUTION: All imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
print('✅ All imports successful!')

In [ ]:
# SOLUTION: Load Titanic data
df = sns.load_dataset('titanic')
print(f'✅ Data loaded! Shape: {df.shape}')
df.head()

In [ ]:
# SOLUTION: Explore data
print('Dataset Information:')
df.info()
print('\nBasic Statistics:')
print(df.describe())
print('\nMissing Values:')
print(df.isnull().sum())

**Question 1.1 ANSWER:** Is this dataset balanced or imbalanced?

*Answer:* **Imbalanced** - About 62% died vs 38% survived (roughly 3:2 ratio). This means accuracy alone could be misleading. We need to focus on precision, recall, and F1-score.

## Feature Engineering - SOLUTION

In [ ]:
# SOLUTION: Select features
features_to_keep = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df_clean = df[features_to_keep].copy()
print(f'✅ Selected features! Shape: {df_clean.shape}')

In [ ]:
# SOLUTION: Handle missing values
df_clean['age'].fillna(df_clean['age'].median(), inplace=True)
df_clean['embarked'].fillna(df_clean['embarked'].mode()[0], inplace=True)
df_clean = df_clean.dropna()
print(f'✅ Missing values handled! Remaining rows: {len(df_clean)}')

In [ ]:
# SOLUTION: Encode categorical variables
df_clean['sex'] = (df_clean['sex'] == 'male').astype(int)
df_clean = pd.get_dummies(df_clean, columns=['embarked'], drop_first=True)
print('✅ Categorical variables encoded!')
print(f'Final columns: {list(df_clean.columns)}')

In [ ]:
# SOLUTION: Separate features and target
X = df_clean.drop('survived', axis=1).values
y = df_clean['survived'].values
print(f'✅ X shape: {X.shape}, y shape: {y.shape}')

## Part 2: Data Preparation - SOLUTION

In [ ]:
# SOLUTION: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'✅ Training set: {X_train.shape}, Test set: {X_test.shape}')

In [ ]:
# SOLUTION: Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'✅ Features scaled! Mean: {X_train_scaled[:, 0].mean():.4f}, Std: {X_train_scaled[:, 0].std():.4f}')

## Part 3: Model Training - SOLUTION

In [ ]:
# SOLUTION: Create and train model
model = LogisticRegression(max_iter=10000, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)
print('✅ Model trained and predictions made!')

## Part 4: Model Evaluation - SOLUTION

In [ ]:
# SOLUTION: Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print('='*60)
print('MODEL PERFORMANCE METRICS')
print('='*60)
print(f'Accuracy:  {accuracy:.3f}')
print(f'Precision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')
print(f'F1-Score:  {f1:.3f}')
print('='*60)

**Question 4.1 ANSWER:** Which metric is most important?

*Answer:* **Recall** is most important for safety planning. We want to identify all survival patterns. False negatives (predicting death when someone survived) mean we're underestimating safety effectiveness. In cruise safety, it's better to over-invest (FP) than under-prepare (FN).

In [ ]:
# SOLUTION: Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print('Confusion Matrix:')
print(cm)
print(f'\nTN={tn}, FP={fp}, FN={fn}, TP={tp}')

In [ ]:
# SOLUTION: Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix: Titanic Survival')
plt.show()

**Question 4.2 ANSWERS:**

a) TP passengers (typically ~49-52) were correctly identified as survivors

b) FN passengers (typically ~17-20) died but model predicted survival - this is concerning for safety planning

c) **False Negatives (FN) are more concerning** because they mean inadequate safety measures. Predicting survival when death occurred means we're underestimating danger, leading to complacency in safety protocols.

In [ ]:
# SOLUTION: Classification report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

In [ ]:
# SOLUTION: ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba[:, 1])
auc = roc_auc_score(y_test, y_pred_proba[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve: Titanic Survival')
plt.legend()
plt.show()
print(f'AUC Score: {auc:.3f}')

**Question 4.3 ANSWERS:**

a) AUC of ~0.85-0.87 means 85-87% probability model ranks a random survivor higher than a random non-survivor. Strong discrimination ability.

b) **Good to Very Good** (0.8-0.9 range is considered good)

## Part 5: Analysis - SOLUTION

In [ ]:
# SOLUTION: Metrics summary
metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
    'Score': [accuracy, precision, recall, f1, auc],
    'Interpretation': ['Overall correctness', 'Accuracy of survival predictions', 'Percentage of survivors identified', 'Balance of precision/recall', 'Overall discrimination']
})
print(metrics_summary.to_string(index=False))

In [ ]:
# SOLUTION: Subgroup analysis
test_indices = df_clean.index[-len(y_test):]
test_df = df_clean.loc[test_indices].copy()
test_df['predicted'] = y_pred
test_df['correct'] = (test_df['survived'] == test_df['predicted'])
test_df['sex_label'] = test_df['sex'].map({1: 'Male', 0: 'Female'})
print('Accuracy by Passenger Class:')
print(test_df.groupby('pclass')['correct'].agg(['sum', 'count', 'mean']))
print('\nAccuracy by Gender:')
print(test_df.groupby('sex_label')['correct'].agg(['sum', 'count', 'mean']))

**Question 5.1 ANSWERS:**

a) **1st class** passengers are easiest to predict (highest accuracy ~85-90%) due to clear survival patterns and better lifeboat access

b) **Females** are easier to predict (accuracy ~85-90%) due to 'women and children first' protocol creating very clear patterns

c) The 'women and children first' evacuation protocol created strong, predictable patterns. Women had 74% survival rate (easy to predict), while male survival was more random at 19% (harder to predict). Similarly, 1st class had clear survival advantages while 3rd class had chaotic evacuation.

## Part 6: Critical Thinking - SOLUTIONS

**Question 6.1a:** Which metric to emphasize?

*Answer:* **Recall** - percentage of survivors identified. False negatives reveal gaps in understanding survival patterns. Also discuss precision to avoid overconfidence.

**Question 6.1b:** Explain false negative?

*Answer:* 'A false negative is like a safety inspector missing a working emergency exit. Our model predicted someone would die, but they survived - perhaps through an unusual evacuation route we didn't account for. Each false negative represents a survival method we're missing in our safety analysis.'

**Question 6.1c:** Safety recommendations?

*Answer:* 1) Equitable lifeboat access (address class disparity), 2) Enhanced family evacuation training, 3) Age-appropriate protocols, 4) Investigate the 20% we got wrong, 5) Sufficient lifeboats for 100% capacity.

**Question 6.2a:** Lower threshold effect?

*Answer:* Recall INCREASES (catch more survivors), Precision DECREASES (more false alarms). Lower threshold = more 'survived' predictions.

**Question 6.2b:** Recommend threshold?

*Answer:* **Lower threshold (0.3)** for safety. Better to over-prepare (accept more FP) than miss survival patterns (FN). Lives matter more than cost.

**Question 6.3a:** Missing factors?

*Answer:* Swimming ability, physical fitness, cabin location, where passenger was when iceberg hit, language barriers, luck, behavioral factors, social factors.

**Question 6.3b:** Improvements?

*Answer:* Add cabin location data, family composition details, crew interaction info, physical characteristics, behavioral data. Try non-linear models (Decision Trees - Week 3!). Create interaction features like 'female_first_class'.

## Part 7: Bonus Visualization - SOLUTION

In [ ]:
# SOLUTION: Probability distribution
prob_survived = y_pred_proba[:, 1]
prob_died_actual = prob_survived[y_test == 0]
prob_survived_actual = prob_survived[y_test == 1]
plt.figure(figsize=(10, 6))
plt.hist(prob_died_actual, bins=20, alpha=0.6, label='Actual: Died', color='red')
plt.hist(prob_survived_actual, bins=20, alpha=0.6, label='Actual: Survived', color='green')
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold 0.5')
plt.xlabel('Predicted Probability of Survival')
plt.ylabel('Frequency')
plt.title('Probability Distribution by Actual Outcome')
plt.legend()
plt.show()

---

## 🎉 Congratulations!

**Key Takeaways:**

1. **Feature engineering matters** - encoding categorical variables correctly is crucial
2. **Context drives metric choice** - safety contexts prioritize recall
3. **Always analyze subgroups** - model performs differently on different types
4. **Models reflect historical patterns** - Titanic model captures 'women and children first'

**Next Steps:**
1. Complete self-assessment
2. Review confusion matrix until you can draw it from memory
3. Practice explaining precision vs recall
4. Get ready for Week 3 - Decision Trees!

---

*Week 2 Post-Class SOLUTIONS v1.0 | December 2025*